In [1]:
import json
from PIL import Image, ImageDraw, ImageFont
import subprocess

In [2]:
#testing only
def generate_sample_timeline():
    WIDTH = 20000
    HEIGHT = 1080
    BLOCK_WIDTH = 600
    START_X = 400

    with open("data.json") as f:
        movies = json.load(f)

    img = Image.new("RGB",(WIDTH,HEIGHT),(15,15,15))
    draw = ImageDraw.Draw(img)

    title_font = ImageFont.truetype("Arial.ttf",70)
    text_font = ImageFont.truetype("Arial.ttf",40)

    draw.text((WIDTH/2-400,50),
            "Tom Cruise Movie Salaries Timeline",
            fill="white",
            font=title_font)

    timeline_y = HEIGHT//2

    draw.line((0,timeline_y,WIDTH,timeline_y),fill="white",width=6)

    x = START_X

    for m in movies:

        center = x + BLOCK_WIDTH//2

        draw.line((center,timeline_y-120,center,timeline_y+120),
                fill="white",width=5)

        draw.text((x,timeline_y-200),
                m["movie"],
                fill="white",
                font=text_font)

        draw.text((x,timeline_y-140),
                str(m["year"]),
                fill="gray",
                font=text_font)

        draw.text((x,timeline_y+150),
                m["salary"],
                fill="yellow",
                font=text_font)

        x += BLOCK_WIDTH + 120

    img.save("timeline.png")

    print("timeline.png created")

In [3]:
def generate_sample_video():
    duration = 480
    video_w = 1080

    positions = [0,800,1600,2400,3200,4000]

    pause = 3
    scroll_time = 8

    segments = []
    time = 0

    for i,p in enumerate(positions):

        segments.append(f"if(lt(t,{time+scroll_time}),{p},")
        time += scroll_time

        segments.append(f"if(lt(t,{time+pause}),{p},")
        time += pause

    segments.append("0" + ")"*len(segments))

    expr = "".join(segments)

    cmd = [ #
    "ffmpeg",
    "-loop","1",
    "-t",str(duration),
    "-i","timeline.png",
    "-vf",f"crop=1080:1920:x='{expr}':y=0",
    "-r","30",
    "-pix_fmt","yuv420p",
    "-c:v","libx264",
    "timeline_video.mp4"
    ]

    subprocess.run(cmd)

ffmpeg -loop 1 -t 480 -i timeline.png \
-vf "crop=1920:1080:x='(iw-1920)*t/480':y=0" \
-r 90 -pix_fmt yuv420p -c:v libx264 timeline_video.mp4

<h2>Download a part of video from youtube</h2>
#python3.10 -m pip install yt-dlp

yt-dlp \
--download-sections "*00:00:40-00:01:00" \
--force-keyframes-at-cuts \
-f "bv*+ba/b" \
--merge-output-format mp4 \
-o "maverick.mp4" \
"https://www.youtube.com/watch?v=qSqVVswa420"